In [ ]:
import requests
import pandas as pd
import time

def fetch_kyobo_authors(pages: int = 1):
    print("===> 작가 정보 수집 시작")
    author_detail_list = []

    base_api = "https://store.kyobobook.co.kr/api/gw/best/best-seller/author"
    base_img_url = "https://contents.kyobobook.co.kr/sih/fit-in/200x0/dtl/author/"
    base_detail_url = "https://store.kyobobook.co.kr/person/detail/"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Referer": "https://store.kyobobook.co.kr/bestseller/person/daily/domestic",
        "Accept": "application/json, text/plain, */*"
    }

    for page in range(1, pages + 1):
        params = {
            "page": page,
            "per": 20,
            "period": "001",  # 일간
            "dsplDvsnCode": "001",
            "dsplTrgtDvsnCode": "002"
        }

        res = requests.get(base_api, headers=headers, params=params)
        print(f"{res.ok} - {page}페이지 요청됨")

        if not res.ok:
            print(f"❌ 요청 실패: {res.status_code}")
            continue

        try:
            data = res.json()
            writers = data.get("data", {}).get("writerList", [])
        except Exception as e:
            print(f"❌ JSON 파싱 오류: {e}")
            continue

        for idx, item in enumerate(writers, 1):
            try:
                info = item.get("writerInfo", {})
                name = info.get("chrcName", "").strip()
                chrCd = item.get("chrCd", "").strip()  # <- 핵심 ID

                # 대표 도서
                book_titles = [
                    b.get("cmdtName", "").strip()
                    for b in item.get("representativeBook", [])
                    if b.get("cmdtName")
                ]
                books = ", ".join(book_titles[:3])  # 최대 3권

                # 이미지, 상세링크 구성
                image_url = f"{base_img_url}{chrCd}.jpg" if chrCd else "https://via.placeholder.com/120?text=No+Image"
                detail_url = f"{base_detail_url}{chrCd}" if chrCd else "#"

                author_detail = {
                    "순위": idx + (page - 1) * 20,
                    "이름": name,
                    "대표도서": books,
                    "chrCd": chrCd,
                    "이미지": image_url,
                    "상세링크": detail_url
                }

                author_detail_list.append(author_detail)
                print(f"[{author_detail['순위']}] {name} - chrCd: {chrCd}")

            except Exception as e:
                print(f"❌ 작가 처리 오류: {e}")
                continue

        time.sleep(0.5)

    print(f"===> 수집 완료. 총 {len(author_detail_list)}명")
    return pd.DataFrame(author_detail_list)

df = fetch_kyobo_authors(pages=2)
print(df[['이름', 'chrCd', '이미지']].head())

# 저장도 가능
# df.to_csv("kyobo_authors.csv", index=False)



===> 작가 정보 수집 시작
True - 1페이지 요청됨
[1] 홍범준 - chrCd: 
[2] 이재명 - chrCd: 
[3] 한강 - chrCd: 
[4] 김영하 - chrCd: 
[5] David Cho - chrCd: 
[6] 김종원 - chrCd: 
[7] 인생 녹음 중 - chrCd: 
[8] 최태성 - chrCd: 
[9] 양귀자 - chrCd: 
[10] 백난도 - chrCd: 
[11] 전승환 - chrCd: 
[12] 김주완 - chrCd: 
[13] 코이케 류노스케 - chrCd: 
[14] 이선 몰릭 - chrCd: 
[15] 안시내 - chrCd: 
[16] 존 윌리엄스 - chrCd: 
[17] 헤르만 헤세 - chrCd: 
[18] 유발 하라리 - chrCd: 
[19] 김동연 - chrCd: 
[20] 백온유 - chrCd: 
True - 2페이지 요청됨
[21] 태수 - chrCd: 
[22] 김재철 - chrCd: 
[23] 인이이 - chrCd: 
[24] 최용준 - chrCd: 
[25] 김주환 - chrCd: 
[26] 이홍섭 - chrCd: 
[27] 이주윤 - chrCd: 
[28] 서주희 - chrCd: 
[29] 외르크 베르나르디 - chrCd: 
[30] 박준 - chrCd: 
[31] 신민숙 - chrCd: 
[32] 최유리 - chrCd: 
[33] 히가시노 게이고 - chrCd: 
[34] 조연심 - chrCd: 
[35] 이선재 - chrCd: 
[36] 아이다이로 - chrCd: 
[37] 조앤 K. 롤링 - chrCd: 
[38] 김기훈 - chrCd: 
[39] 임주영 - chrCd: 
[40] 전용문 - chrCd: 
===> 수집 완료. 총 40명
          이름 chrCd                                            이미지
0        홍범준        https://via.placeholder.com/120?text=No+Image
1        

,name,chrcId,books,image,link
0,이재명,,"결국 국민이 합니다, 함께 가는 길은 외롭지 않습니다, 그 꿈이 있어 여기까지 왔다",https://via.placeholder.com/120?text=No+Image,#
1,홍범준,,"쎈B 초등 수학 3-2(2025), 일품 중등 수학 1-1 462제(2025), 쎈...",https://via.placeholder.com/120?text=No+Image,#
2,한강,,"흰, Lecons de Grec, Ces soirs ranges dans mon t...",https://via.placeholder.com/120?text=No+Image,#
3,김영하,,"단 한 번의 삶, 김영하의 세계문학 원정대 4: 빨간 머리 앤, 김영하의 세계문학 ...",https://via.placeholder.com/120?text=No+Image,#
4,David Cho,,해커스 토플 스피킹 인터미디엇(Hackers TOEFL Speaking Interm...,https://via.placeholder.com/120?text=No+Image,#
5,최태성,,"최태성의 한능검 한국사 6, 최태성의 한능검 한국사: 5 신라·가야, 큰별쌤 최태성...",https://via.placeholder.com/120?text=No+Image,#
6,백온유,,"제16회 젊은작가상 수상작품집(2025), 소설의 첫 만남: 첫사랑 세트, 정원에 대하여",https://via.placeholder.com/120?text=No+Image,#
7,양귀자,,"사피엔스 한국문학 중 단편소설 세트, 모순, 식구·소음공해",https://via.placeholder.com/120?text=No+Image,#
8,김주완,,"줬으면 그만이지(반양장), 지역출판으로 먹고살 수 있을까(큰글자책), 지역출판으로 ...",https://via.placeholder.com/120?text=No+Image,#
9,김종원,,"아이의 어휘력을 위한 66일 필사 노트, 태어나려는 자는 하나의 세계를 깨뜨려야 한...",https://via.placeholder.com/120?text=No+Image,#
